 # Librerias

 

In [1]:
import pandas as pd
import ast
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

 # Carga de datos 


In [2]:
dfentero = pd.read_csv(r"..\..\Data\clean_data_24-03-2026.csv",parse_dates=['insert_date','first_review_date','last_review_date'])

In [3]:
dfentero.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 9650 non-null   int64         
 1   name                         9647 non-null   object        
 2   description                  9650 non-null   object        
 3   host_id                      9650 non-null   int64         
 4   neighbourhood_name           9650 non-null   object        
 5   neighbourhood_district       9650 non-null   object        
 6   room_type                    9650 non-null   object        
 7   accommodates                 9650 non-null   int64         
 8   bathrooms                    9576 non-null   float64       
 9   bedrooms                     9581 non-null   float64       
 10  beds                         9605 non-null   float64       
 11  amenities_list               9650 non-null 

In [4]:
df=dfentero[['apartment_id','neighbourhood_name','minimum_nights','maximum_nights','review_scores_rating','review_scores_location','city']].copy()

In [5]:
df.head(5)

,apartment_id,neighbourhood_name,minimum_nights,maximum_nights,review_scores_rating,review_scores_location,city
0,11964,Centro,3,365,97.0,100.0,Malaga
1,21853,C�rmenes,4,40,92.0,80.0,Madrid
2,32347,San Vicente,2,120,98.0,100.0,Sevilla
3,35379,l'Antiga Esquerra de l'Eixample,2,730,94.0,100.0,Barcelona
4,35801,Quart,1,180,97.0,100.0,Girona


In [6]:
listabarrios=df['neighbourhood_name'].unique().tolist()

In [39]:
df['review_scores_location'].mean()

np.float64(95.43273350471294)

In [7]:
listabarrios

['Centro',
 'C�rmenes',
 'San Vicente',
 "l'Antiga Esquerra de l'Eixample",
 'Quart',
 'Torroella de Fluvi�',
 "el Camp de l'Arpa del Clot",
 "la Dreta de l'Eixample",
 'Embajadores',
 "el Camp d'en Grassot i Gr�cia Nova",
 'el Raval',
 'el Fort Pienc',
 'Palacio',
 'Palomeras Bajas',
 'Lloret de Mar',
 'EL PILAR',
 'Vallvidrera, el Tibidabo i les Planes',
 'Sants',
 'Sant Antoni',
 'Forallac',
 'les Corts',
 'Palma de Mallorca',
 'Universidad',
 'Alc�dia',
 'Justicia',
 'Horta',
 'Sant Pere, Santa Caterina i la Ribera',
 'el Poble Sec',
 'EN CORTS',
 'Capmany',
 'Aluche',
 "Castell� d'Emp�ries",
 'Can Peguera',
 'Cortes',
 'Ciudad Jard�n',
 'la Vila de Gr�cia',
 "la Nova Esquerra de l'Eixample",
 'Ni�o Jes�s',
 'Sol',
 'RUSSAFA',
 'Vilapicina i la Torre Llobeta',
 'Arenal',
 'Begur',
 'Selva',
 'Alfalfa',
 'la Sagrada Fam�lia',
 'Santa Margalida',
 'EL CARME',
 'Tossa de Mar',
 'el Putxet i el Farr�',
 'el Barri G�tic',
 'Es Mercadal',
 'el Poblenou',
 'Campos',
 'S�ller',
 'Ciutadell

In [8]:
# 1. Pasamos todo a minúsculas y quitamos los espacios de los extremos
df['neighbourhood_name'] = df['neighbourhood_name'].str.lower().str.strip()



In [23]:
# Filtramos exigiendo que las noches sean mayores de 30 Y menores o iguales a 341
df_temporada = df[(df['minimum_nights'] > 30) & (df['maximum_nights'] <= 341)].copy()

# Si quieres ver cuántos pisos te han quedado en este segmento exacto:
print("Número de pisos en este rango:", len(df_temporada))

# Ver los primeros datos para comprobarlo
df_temporada[['neighbourhood_name', 'minimum_nights', 'maximum_nights']].head()

Número de pisos en este rango: 110


,neighbourhood_name,minimum_nights,maximum_nights
32,"sant pere, santa caterina i la ribera",31,150
37,el raval,32,330
138,la prosperitat,90,180
200,sol,90,300
356,el parc i la llacuna del poblenou,31,180


In [21]:
df['maximum_nights'].value_counts()

maximum_nights
1125    5913
30       554
365      269
60       224
1124     200
        ... 
1036       1
179        1
1022       1
399        1
105        1
Name: count, Length: 147, dtype: int64

In [9]:
df['neighbourhood_name'].value_counts()

neighbourhood_name
centro                    325
la dreta de l'eixample    297
embajadores               278
palma de mallorca         209
el raval                  206
                         ... 
riudarenes                  1
ripoll                      1
setcases                    1
peralada                    1
darnius                     1
Name: count, Length: 520, dtype: int64

In [10]:
# Ambas condiciones tienen que cumplirse para que sea True
df['es_turistico'] = (df['minimum_nights'] <= 31) & (df['maximum_nights'] <= 31)

In [11]:
df['es_turistico'].value_counts()

es_turistico
False    7634
True     2016
Name: count, dtype: int64

In [19]:


# 1. SELECCIÓN DE VARIABLES Y LIMPIEZA DE NULOS
# Scikit-learn dará un error fatal si hay valores vacíos (NaN) en estas columnas. 
# Los rellenamos con la mediana del mercado para no alterar el modelo.
features = ['review_scores_rating', 'review_scores_location', 'minimum_nights', 'maximum_nights']
df_ml = df.dropna().copy()



X = df_ml[features]

# 2. PREPROCESAMIENTO: ESCALADO (MinMaxScaler)
# Aplastamos las métricas para que todas valgan entre 0 y 1.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 3. CREACIÓN DEL "PISO SINTÉTICO IDEAL"
# Al usar pd.DataFrame le pasamos los nombres de las columnas para que scikit-learn no se queje
piso_ideal = pd.DataFrame([[100, 100, 2, 31]], columns=features)

# Transformamos este piso ficticio usando el mismo escalador que los datos reales
piso_ideal_scaled = scaler.transform(piso_ideal)

# 4. ENTRENAMIENTO DEL MODELO DE MACHINE LEARNING
# Usamos NearestNeighbors para mapear todos los apartamentos en un espacio vectorial
modelo_nn = NearestNeighbors(metric='euclidean')
modelo_nn.fit(X_scaled) # Aquí es donde el modelo "aprende" la distribución

# 5. CÁLCULO DE DISTANCIAS AL PUNTO IDEAL
# Le pedimos al modelo la distancia euclidiana desde el piso ideal a TODOS los pisos reales
distancias, indices = modelo_nn.kneighbors(piso_ideal_scaled, n_neighbors=len(X_scaled))

# 6. ASIGNACIÓN DE RESULTADOS AL DATAFRAME
# Las distancias vienen ordenadas, así que las emparejamos con su índice correcto
serie_distancias = pd.Series(distancias[0], index=df_ml.index[indices[0]])
df_ml['Score_Optimizacion'] = serie_distancias

# 7. AGRUPACIÓN POR CIUDAD Y BARRIO
# Le pasamos una lista con las dos columnas ['city', 'neighbourhood_name']
ranking_ml = df_ml.groupby(['city', 'neighbourhood_name'])['Score_Optimizacion'].median().reset_index()

# Ordenamos de mayor a menor potencial
ranking_ml = ranking_ml.sort_values('Score_Optimizacion', ascending=False)

# Vemos el Top 10 con su ciudad al lado
ranking_ml.head(10)

,city,neighbourhood_name,Score_Optimizacion
181,Girona,vilaju�ga,1.156701
400,Sevilla,"prado, parque mar�a luisa",1.122422
435,Valencia,el calvari,1.052373
32,Barcelona,la marina del prat vermell,1.041708
460,Valencia,la vega baixa,1.037720
153,Girona,sant ferriol,1.035534
7,Barcelona,el bon pastor,1.025641
447,Valencia,favara,1.022891
373,Sevilla,"cruz roja, capuchinos",1.017958
390,Sevilla,"la palmilla, doctor mara��n",1.016037


In [20]:
# 1. Calculamos las distancias individuales (los "sub-apartados") para saber POR QUÉ fallan
df_ml['dist_rating'] = 100 - df_ml['review_scores_rating']
df_ml['dist_location'] = 100 - df_ml['review_scores_location']
df_ml['dist_min_nights'] = abs(df_ml['minimum_nights'] - 2)
df_ml['dist_max_nights'] = abs(df_ml['maximum_nights'] - 31)

# 2. Elegimos las columnas que queremos ver en la tabla final
columnas_a_mostrar = [
    'dist_rating', 
    'dist_location', 
    'dist_min_nights', 
    'dist_max_nights', 
    'Score_Optimizacion' # Mantenemos la nota de la IA al final
]

# 3. Agrupamos por ciudad y barrio, sacando la mediana de todo
ranking_detallado = df_ml.groupby(['city', 'neighbourhood_name'])[columnas_a_mostrar].median().reset_index()

# 4. ORDENAMOS usando el algoritmo de Machine Learning (Score_Optimizacion)
ranking_detallado = ranking_detallado.sort_values('Score_Optimizacion', ascending=False)

# 5. Parche estético para arreglar los nombres rotos que vimos en tu captura
parches_esteticos = {
    'vilajuga': 'vilajuïga',
    'prado, parque mara luisa': 'prado, parque maría luisa',
    'la palmilla, doctor maran': 'la palmilla, doctor marañón',
    'opael': 'opañel',
    'el calvari': 'el calvari', # Ya está en minúscula, pero por si acaso
    'la vega baixa': 'la vega baixa'
}
ranking_detallado['neighbourhood_name'] = ranking_detallado['neighbourhood_name'].replace(parches_esteticos)

# 6. Vemos el Top 10 con todo el detalle de la IA + las explicaciones
ranking_detallado.head(10)

,city,neighbourhood_name,dist_rating,dist_location,dist_min_nights,dist_max_nights,Score_Optimizacion
181,Girona,vilaju�ga,30.0,40.0,1.0,1094.0,1.156701
400,Sevilla,"prado, parque mar�a luisa",40.0,20.0,0.0,1094.0,1.122422
435,Valencia,el calvari,25.0,20.0,1.0,1094.0,1.052373
32,Barcelona,la marina del prat vermell,16.0,30.0,1.0,1069.0,1.041708
460,Valencia,la vega baixa,27.0,10.0,0.0,1094.0,1.037720
153,Girona,sant ferriol,20.0,20.0,0.0,1094.0,1.035534
7,Barcelona,el bon pastor,20.5,5.0,0.5,1094.0,1.025641
447,Valencia,favara,16.5,10.0,0.0,1094.0,1.022891
373,Sevilla,"cruz roja, capuchinos",13.0,20.0,0.0,1094.0,1.017958
390,Sevilla,"la palmilla, doctor mara��n",12.0,20.0,0.0,1094.0,1.016037


In [28]:

# 1. COPIA Y LIMPIEZA DE NULOS (Eliminamos en lugar de rellenar)
df_final = df.copy()

# Definimos las columnas clave que el modelo necesita sí o sí
todas_las_features = ['review_scores_rating', 'minimum_nights', 'maximum_nights', 'review_scores_location']

# ELIMINAMOS las filas que tengan algún nulo en estas columnas (no las tenemos en cuenta)
df_final = df_final.dropna(subset=todas_las_features).copy()

# ==========================================
# MODELO 1: POTENCIAL DE CALIDAD (Solo Reviews)
# ==========================================
feat_calidad = ['review_scores_rating']

scaler_calidad = MinMaxScaler()
X_calidad = scaler_calidad.fit_transform(df_final[feat_calidad])

# El ideal es 100 de nota
ideal_calidad = pd.DataFrame([[100]], columns=feat_calidad)
ideal_calidad_scaled = scaler_calidad.transform(ideal_calidad)

modelo_calidad = NearestNeighbors(metric='euclidean')
modelo_calidad.fit(X_calidad)
dist_calidad, ind_calidad = modelo_calidad.kneighbors(ideal_calidad_scaled, n_neighbors=len(X_calidad))

# Guardamos el resultado del Modelo 1
df_final['Score_Calidad'] = pd.Series(dist_calidad[0], index=df_final.index[ind_calidad[0]])

# ==========================================
# MODELO 2: POTENCIAL LEGAL/GESTIÓN (Solo Noches)
# ==========================================
feat_noches = ['minimum_nights', 'maximum_nights']

scaler_noches = MinMaxScaler()
X_noches = scaler_noches.fit_transform(df_final[feat_noches])

# El ideal es 2 noches mínimas y 31 máximas
ideal_noches = pd.DataFrame([[2, 31]], columns=feat_noches)
ideal_noches_scaled = scaler_noches.transform(ideal_noches)

modelo_noches = NearestNeighbors(metric='euclidean')
modelo_noches.fit(X_noches)
dist_noches, ind_noches = modelo_noches.kneighbors(ideal_noches_scaled, n_neighbors=len(X_noches))

# Guardamos el resultado del Modelo 2
df_final['Score_Legal'] = pd.Series(dist_noches[0], index=df_final.index[ind_noches[0]])

# ==========================================
# MODELO 3: POTENCIAL DE LOCALIZACIÓN (Expectativas)
# ==========================================
feat_loc = ['review_scores_location']

scaler_loc = MinMaxScaler()
X_loc = scaler_loc.fit_transform(df_final[feat_loc])

# El ideal es 100 de nota en localización
ideal_loc = pd.DataFrame([[100]], columns=feat_loc)
ideal_loc_scaled = scaler_loc.transform(ideal_loc)

modelo_loc = NearestNeighbors(metric='euclidean')
modelo_loc.fit(X_loc)
dist_loc, ind_loc = modelo_loc.kneighbors(ideal_loc_scaled, n_neighbors=len(X_loc))

# Guardamos el resultado del Modelo 3
df_final['Score_Localizacion'] = pd.Series(dist_loc[0], index=df_final.index[ind_loc[0]])

# ==========================================
# AGRUPACIÓN MAESTRA Y RESULTADOS FINALES
# ==========================================
columnas_a_mostrar = [
    'Score_Calidad', 
    'Score_Legal', 
    'Score_Localizacion', 
    'review_scores_rating', 
    'minimum_nights', 
    'maximum_nights', 
    'review_scores_location'
]

# Agrupamos por ciudad y barrio una ÚNICA vez
ranking_unificado = df_final.groupby(['city', 'neighbourhood_name'])[columnas_a_mostrar].median().reset_index()

# Parche estético para los nombres
parches_esteticos = {
    'vilajuga': 'vilajuïga',
    'prado, parque mara luisa': 'prado, parque maría luisa',
    'la palmilla, doctor maran': 'la palmilla, doctor marañón',
    'opael': 'opañel',
    'el calvari': 'el calvari',
    'la vega baixa': 'la vega baixa'
}
ranking_unificado['neighbourhood_name'] = ranking_unificado['neighbourhood_name'].replace(parches_esteticos)


# ==========================================
# VISUALIZACIÓN DE LOS 3 TOP 5 (Misma tabla unificada)
# ==========================================
print("--- TOP 5: POTENCIAL DE OPTIMIZACIÓN POR CALIDAD (Peores reseñas) ---")
display(ranking_unificado.sort_values('Score_Calidad', ascending=False).head(5))

print("\n")

print("--- TOP 5: POTENCIAL DE OPTIMIZACIÓN LEGAL (Malas políticas de noches) ---")
display(ranking_unificado.sort_values('Score_Legal', ascending=False).head(5))

print("\n")

print("--- TOP 5: POTENCIAL DE OPTIMIZACIÓN POR LOCALIZACIÓN (Expectativas / Marketing) ---")
display(ranking_unificado.sort_values('Score_Localizacion', ascending=False).head(5))

--- TOP 5: POTENCIAL DE OPTIMIZACIÓN POR CALIDAD (Peores reseñas) ---


,city,neighbourhood_name,Score_Calidad,Score_Legal,Score_Localizacion,review_scores_rating,minimum_nights,maximum_nights,review_scores_location
400,Sevilla,"prado, parque mar�a luisa",0.5000,0.973310,0.250,60.0,2.0,1125.0,80.0
188,Madrid,aguilas,0.5000,0.024087,0.250,60.0,2.0,12.0,80.0
181,Girona,vilaju�ga,0.3750,0.973310,0.500,70.0,1.0,1125.0,60.0
272,Madrid,rosas,0.3750,0.021371,0.250,70.0,1.0,7.0,80.0
130,Girona,osor,0.3375,0.000890,0.375,73.0,1.0,31.0,70.0




--- TOP 5: POTENCIAL DE OPTIMIZACIÓN LEGAL (Malas políticas de noches) ---


,city,neighbourhood_name,Score_Calidad,Score_Legal,Score_Localizacion,review_scores_rating,minimum_nights,maximum_nights,review_scores_location
41,Barcelona,la trinitat vella,0.00,0.991041,0.125,100.0,150.5,1125.0,90.0
415,Sevilla,"tabladilla, la estrella",0.05,0.973788,0.125,96.0,60.0,1124.0,90.0
37,Barcelona,la sagrera,0.00,0.973652,0.000,100.0,31.0,1125.0,100.0
173,Girona,vall-llobrega,0.00,0.973320,0.000,100.0,7.0,1125.0,100.0
339,Mallorca,porreres,0.00,0.973316,0.000,100.0,6.0,1125.0,100.0




--- TOP 5: POTENCIAL DE OPTIMIZACIÓN POR LOCALIZACIÓN (Expectativas / Marketing) ---


,city,neighbourhood_name,Score_Calidad,Score_Legal,Score_Localizacion,review_scores_rating,minimum_nights,maximum_nights,review_scores_location
181,Girona,vilaju�ga,0.3750,0.973310,0.500,70.0,1.0,1125.0,60.0
385,Sevilla,la bachillera,0.1250,0.021352,0.375,90.0,2.0,7.0,70.0
32,Barcelona,la marina del prat vermell,0.2000,0.951068,0.375,84.0,3.0,1100.0,70.0
130,Girona,osor,0.3375,0.000890,0.375,73.0,1.0,31.0,70.0
373,Sevilla,"cruz roja, capuchinos",0.1625,0.973310,0.250,87.0,2.0,1125.0,80.0


In [30]:
# ==========================================
# FASE 1: VISIÓN MACRO (Análisis a nivel Ciudad)
# ==========================================
columnas_macro = ['Score_Calidad', 'Score_Legal', 'Score_Localizacion']

# Agrupamos SOLO por ciudad calculando la mediana
ranking_ciudades = df_final.groupby('city')[columnas_macro].mean().reset_index()

# Mostramos los rankings a nivel macro
print("🏙️ --- VISIÓN MACRO 1: CIUDADES CON PEOR CALIDAD GENERAL ---")
display(ranking_ciudades.sort_values('Score_Calidad', ascending=False)[['city', 'Score_Calidad']])
print("\n")

print("⚖️ --- VISIÓN MACRO 2: CIUDADES CON PEOR GESTIÓN LEGAL (Noches) ---")
display(ranking_ciudades.sort_values('Score_Legal', ascending=False)[['city', 'Score_Legal']])
print("\n")

print("📍 --- VISIÓN MACRO 3: CIUDADES CON PEOR LOCALIZACIÓN (Expectativas) ---")
display(ranking_ciudades.sort_values('Score_Localizacion', ascending=False)[['city', 'Score_Localizacion']])

🏙️ --- VISIÓN MACRO 1: CIUDADES CON PEOR CALIDAD GENERAL ---


,city,Score_Calidad
0,Barcelona,0.115705
1,Girona,0.111389
3,Malaga,0.097500
7,Valencia,0.097371
2,Madrid,0.092212
4,Mallorca,0.087667
5,Menorca,0.084201
6,Sevilla,0.080738




⚖️ --- VISIÓN MACRO 2: CIUDADES CON PEOR GESTIÓN LEGAL (Noches) ---


,city,Score_Legal
4,Mallorca,0.719387
5,Menorca,0.693222
3,Malaga,0.667832
2,Madrid,0.646666
6,Sevilla,0.629638
1,Girona,0.628419
0,Barcelona,0.603462
7,Valencia,0.587131




📍 --- VISIÓN MACRO 3: CIUDADES CON PEOR LOCALIZACIÓN (Expectativas) ---


,city,Score_Localizacion
4,Mallorca,0.069516
1,Girona,0.068611
3,Malaga,0.065663
0,Barcelona,0.058313
5,Menorca,0.057292
7,Valencia,0.056075
2,Madrid,0.044306
6,Sevilla,0.040398
